In [1]:
import pandas as pd
import geopandas as gpd
import typing as T
from shapely.geometry import Point

In [2]:
# -------------------------
# CRS handling (WGS84 -> UTM 51N)
# -------------------------

SOURCE_CRS = "EPSG:4326"     # WGS 84
ANALYSIS_CRS = "EPSG:32651"  # WGS 84 / UTM zone 51N

def to_geodataframe(
    df_with_coords: pd.DataFrame,
    lon_col: str = "lon",
    lat_col: str = "lat",
    source_crs: str = SOURCE_CRS,
    target_crs: T.Optional[str] = ANALYSIS_CRS,
) -> gpd.GeoDataFrame:
    gdf = gpd.GeoDataFrame(
        df_with_coords.copy(),
        geometry=[
            Point(xy) if pd.notna(xy[0]) and pd.notna(xy[1]) else None
            for xy in zip(df_with_coords[lon_col], df_with_coords[lat_col])
        ],
        crs=source_crs
    )
    gdf = gdf[~gdf["geometry"].isna()].copy()
    if target_crs:
        gdf = gdf.to_crs(target_crs)
    return gdf

In [3]:
df_listings = pd.read_csv("listings-cleaned.csv")
df_listings.head()

,project_name,floor_area,price,bedrooms,bathrooms,condition,parking,price_per_sqm
0,The Calinea Tower,126,22000000.0,3,3,Unfurnished,0.0,174603.174603
1,The Calinea Tower,86,13025000.0,3,2,Unfurnished,0.0,151453.488372
2,The Calinea Tower,30,6007000.0,1,1,Unfurnished,0.0,200233.333333
3,The Calinea Tower,70,10938000.0,2,1,Unfurnished,0.0,156257.142857
4,The Calinea Tower,79,11700000.0,3,2,Unfurnished,0.0,148101.265823


In [4]:
df_properties = pd.read_csv("geocoded-properties.csv", dtype=str, encoding='utf-8-sig')
df_properties.head()

,project_name,location,full_address,cleaned_address,cleaned_query,fallback_query,lat,lon,geocode_provider,formatted_address,quality_note
0,100 West,"Pio Del Pilar, Makati, Metro Manila","100 West, Pio Del Pilar, Makati, Metro Manila,...",100 west pio del pilar makati metro manila phi...,100 west pio del pilar makati metro manila phi...,"100 West, Pio Del Pilar, Makati, Metro Manila,...",14.5584322,121.0098588,google,"100 Sen. Gil Puyat Ave. Corner Washington St.,...",FALLBACK_FULL|GEOMETRIC_CENTER
1,100 West Makati,"Pio Del Pilar, Makati, Metro Manila","100 West Makati, Pio Del Pilar, Makati, Metro ...",100 west makati pio del pilar makati metro man...,100 west makati pio del pilar makati metro man...,NaN,14.5584322,121.0098588,google,"100 Sen. Gil Puyat Ave. Corner Washington St.,...",CLEANED|GEOMETRIC_CENTER
2,1001 Parkway Residences,"Muntinlupa, Metro Manila","1001 Parkway Residences, Muntinlupa, Metro Man...",1001 parkway muntinlupa metro manila philippines,1001 parkway muntinlupa metro manila philippines,NaN,14.4141232,121.0372049,google,"C27P+JVW, Muntinlupa, Metro Manila, Philippines",CLEANED|GEOMETRIC_CENTER
3,101 Newport BLVD,"Pasay, Metro Manila","101 Newport BLVD, Pasay, Metro Manila, Philipp...",101 newport boulevard pasay metro manila phili...,101 newport boulevard pasay metro manila phili...,NaN,14.5237733,121.0129988,google,"101 Newport Blvd, Newport City, Pasay City, Me...",CLEANED|ROOFTOP
4,101 Xavierville,"Loyola Heights, Quezon City, Metro Manila","101 Xavierville, Loyola Heights, Quezon City, ...",101 xavierville loyola heights quezon city met...,101 xavierville loyola heights quezon city met...,NaN,14.6341152,121.0734137,google,"101 Xavierville Ave, Diliman, Quezon City, 110...",CLEANED|ROOFTOP


In [12]:
# join df_properties to df_listings using the project_name column
df_merged = pd.merge(
    df_listings,
    df_properties[['project_name', 'lat', 'lon']],
    how='left',
    left_on='project_name',
    right_on='project_name',
    suffixes=('', '_property')
)
df_merged.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
project_name,6084,679,Sonora Garden Residences,181,NaN,NaN,NaN,NaN,NaN,NaN,NaN
floor_area,6084.0,NaN,NaN,NaN,69.50526,51.753575,12.0,33.0,54.0,83.25,309.0
price,6084.0,NaN,NaN,NaN,17221373.835306,19611472.564219,1000000.0,5600000.0,9500000.0,20899000.0,135000000.0
bedrooms,6084.0,NaN,NaN,NaN,1.712032,0.733129,1.0,1.0,2.0,2.0,4.0
bathrooms,6084.0,NaN,NaN,NaN,1.483728,0.743362,1.0,1.0,1.0,2.0,4.0
condition,6084,3,Unfurnished,2740,NaN,NaN,NaN,NaN,NaN,NaN,NaN
parking,6084.0,NaN,NaN,NaN,0.311637,0.57699,0.0,0.0,0.0,1.0,2.0
price_per_sqm,6084.0,NaN,NaN,NaN,217784.156622,94611.693297,19157.088123,147207.891667,201801.948052,283783.783784,612903.225806
lat,4673,444,14.4443435,181,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lon,4673,442,120.9992136,181,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# remove rows with missing lat/lon
df_merged = df_merged[~df_merged['lat'].isna() & ~df_merged['lon'].isna()].copy()
df_merged.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
project_name,4673,473,Sonora Garden Residences,181,NaN,NaN,NaN,NaN,NaN,NaN,NaN
floor_area,4673.0,NaN,NaN,NaN,67.171624,49.933034,12.0,33.0,52.0,80.0,309.0
price,4673.0,NaN,NaN,NaN,16332801.682645,18591889.314559,1000000.0,5590000.0,8875600.0,19500000.0,135000000.0
bedrooms,4673.0,NaN,NaN,NaN,1.680933,0.714525,1.0,1.0,2.0,2.0,4.0
bathrooms,4673.0,NaN,NaN,NaN,1.435908,0.699966,1.0,1.0,1.0,2.0,4.0
condition,4673,3,Unfurnished,2261,NaN,NaN,NaN,NaN,NaN,NaN,NaN
parking,4673.0,NaN,NaN,NaN,0.28012,0.554466,0.0,0.0,0.0,0.0,2.0
price_per_sqm,4673.0,NaN,NaN,NaN,215246.428855,93965.810283,19157.088123,145000.0,198130.841121,280701.754386,612903.225806
lat,4673,444,14.4443435,181,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lon,4673,442,120.9992136,181,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df_hospitals = pd.read_csv("geocoded-philhealth-hospitals.csv")
df_hospitals.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
hospital,169,168,"SAN LORENZO HOSPITAL HEALTH\nMANAGEMENT CO., INC.",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
beds,169.0,NaN,NaN,NaN,178.597633,376.71788,6.0,28.0,80.0,200.0,4200.0
category,169,4,LEVEL 3,58,NaN,NaN,NaN,NaN,NaN,NaN,NaN
street,169,167,"EAST AVENUE, DILIMAN,",3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,169,25,QUEZON CITY,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sec,169,2,P,115,NaN,NaN,NaN,NaN,NaN,NaN,NaN
full_address,169,169,ALLIED CARE EXPERTS (ACE) MEDICAL\nCENTER - QU...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cleaned_query,169,169,allied care experts (ace) medical center - que...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fallback_query,134,134,ALLIED CARE EXPERTS (ACE) MEDICAL\nCENTER - QU...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,38.0,NaN,NaN,NaN,14.613572,0.056969,14.427619,14.59279,14.621102,14.648177,14.753278


In [14]:
# remove rows from df_hospitals with missing lat/lon
df_hospitals = df_hospitals[~df_hospitals['lat'].isna() & ~df_hospitals['lon'].isna()].copy()
df_hospitals.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
hospital,38,38,CARDINAL SANTOS MEDICAL CENTER,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
beds,38.0,NaN,NaN,NaN,211.789474,242.486908,12.0,38.0,107.5,299.25,1000.0
category,38,4,LEVEL 3,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
street,38,36,"EAST AVENUE, DILIMAN,",3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,38,17,QUEZON CITY,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sec,38,2,P,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
full_address,38,38,"CARDINAL SANTOS MEDICAL CENTER, 10 WILSON ST.,...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cleaned_query,38,38,cardinal santos medical center 10 wilson stree...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fallback_query,3,3,"MARIKINA ST. VINCENT GENERAL HOSPITAL,\nINC., ...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,38.0,NaN,NaN,NaN,14.613572,0.056969,14.427619,14.59279,14.621102,14.648177,14.753278


In [16]:
df_stations = pd.read_csv("geocoded-train-stations.csv")
df_stations.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
line,56,4,LRT-1,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
station,56,53,Vito Cruz,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
station_name,56,56,MRT-3 North Avenue Station,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,56,53,"Malate, Manila, 1004 Metro Manila",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
full_address,56,56,"MRT-3 North Avenue Station, North Avenue, Quez...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cleaned_query,56,56,mrt-3 north avenue station north avenue quezon...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fallback_query,3,3,"R. Papa Station, Ricardo Papa St, Santa Cruz, ...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,55.0,NaN,NaN,NaN,14.591493,0.04231,14.485318,14.565275,14.601721,14.622323,14.657605
lon,55.0,NaN,NaN,NaN,121.012213,0.030992,120.973277,120.986644,120.999447,121.034262,121.100275
geocode_provider,55,1,google,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# remove rows from df_stations with missing lat/lon
df_stations = df_stations[~df_stations['lat'].isna() & ~df_stations['lon'].isna()].copy()
df_stations.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
line,55,4,LRT-1,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
station,55,52,Gil Puyat,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
station_name,55,55,MRT-3 North Avenue Station,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,55,52,"Malate, Manila, 1004 Metro Manila",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
full_address,55,55,"MRT-3 North Avenue Station, North Avenue, Quez...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cleaned_query,55,55,mrt-3 north avenue station north avenue quezon...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fallback_query,2,2,"R. Papa Station, Ricardo Papa St, Santa Cruz, ...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,55.0,NaN,NaN,NaN,14.591493,0.04231,14.485318,14.565275,14.601721,14.622323,14.657605
lon,55.0,NaN,NaN,NaN,121.012213,0.030992,120.973277,120.986644,120.999447,121.034262,121.100275
geocode_provider,55,1,google,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df_hei = pd.read_csv("geocoded-ched-hei.csv")
df_hei.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
REGION,318,1,13 - Nat. Capital Region,318,NaN,NaN,NaN,NaN,NaN,NaN,NaN
INSTITUTION NAME,318,318,Adamson University,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
INSTITUTION TYPE,318,4,Private,280,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PROVINCE,318,1,Metro Manila,318,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MUNICIPALITY/CITY,314,17,Quezon City,86,NaN,NaN,NaN,NaN,NaN,NaN,NaN
FULL_ADDRESS,318,318,"Adamson University, City of Manila",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cleaned_query,318,318,adamson university city of manila metro manila...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fallback_query,78,78,"AMA Computer University, Quezon City, Metro Ma...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,255.0,NaN,NaN,NaN,14.599041,0.083989,14.373035,14.556506,14.603738,14.656354,14.780545
lon,255.0,NaN,NaN,NaN,121.02272,0.035301,120.944469,120.992919,121.01922,121.05095,121.106375


In [19]:
# remove rows frrom df_hei with missing lat/lon
df_hei = df_hei[~df_hei['lat'].isna() & ~df_hei['lon'].isna()].copy()
df_hei.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
REGION,255,1,13 - Nat. Capital Region,255,NaN,NaN,NaN,NaN,NaN,NaN,NaN
INSTITUTION NAME,255,255,Adamson University,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
INSTITUTION TYPE,255,4,Private,223,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PROVINCE,255,1,Metro Manila,255,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MUNICIPALITY/CITY,251,17,Quezon City,71,NaN,NaN,NaN,NaN,NaN,NaN,NaN
FULL_ADDRESS,255,255,"Adamson University, City of Manila",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cleaned_query,255,255,adamson university city of manila metro manila...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fallback_query,15,15,"St. Louis College-Valenzuela, City of Valenzue...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,255.0,NaN,NaN,NaN,14.599041,0.083989,14.373035,14.556506,14.603738,14.656354,14.780545
lon,255.0,NaN,NaN,NaN,121.02272,0.035301,120.944469,120.992919,121.01922,121.05095,121.106375
